# DermoGraph-XAI — MaxViT-T
**Team 8 | VIT Bhopal | Skin Lesion Classification**

| | |
|---|---|
| Model | MaxViT-Tiny (Multi-Axis Vision Transformer) |
| Dataset | ALL 6 datasets — HAM10000 + PAD-UFES + ISIC2020 + Melanoma + MIDAS |
| Classes | 7 (Melanoma, Nevi, BCC, AK, BKL, DF, Vasc) |
| Loss | Weighted CrossEntropy |
| Optimizer | AdamW + CosineAnnealingLR |

### Benchmark So Far
| Model | Acc | F1 | AUC |
|---|---|---|---|
| VGG16 | 80.48% | 0.7102 | 0.9601 |
| ResNet50 | 87.40% | 0.7261 | 0.9856 |
| DenseNet121 | 87.69% | 0.7663 | 0.9866 |
| **MaxViT-T** | **?** | **?** | **?** |

## Cell 1 — Install & Verify GPU

In [ ]:
!pip install -q timm albumentations

import torch
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"PyTorch : {torch.__version__}")
print("✓ GPU ready" if torch.cuda.is_available() else "✗ NO GPU — enable in Session options")


## Cell 2 — Imports & Config

In [ ]:
import os, json, time, warnings, shutil, sys
warnings.filterwarnings('ignore')
os.environ['NO_ALBUMENTATIONS_UPDATE'] = '1'

import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm

from sklearn.metrics import (f1_score, roc_auc_score,
                              confusion_matrix, classification_report,
                              precision_score, recall_score)

DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT       = '/kaggle/working/'
CACHE_DIR    = '/kaggle/working/cache/'
BATCH_SIZE   = 16
N_CLASSES    = 7
N_EPOCHS     = 50
LR           = 5e-5
WEIGHT_DECAY = 1e-2
PATIENCE     = 15
GRAD_ACCUM   = 8
MODEL_NAME   = 'maxvit_t'
TIMM_STR     = 'maxvit_tiny_tf_224'
MEAN         = [0.485, 0.456, 0.406]
STD          = [0.229, 0.224, 0.225]
CLASS_NAMES  = ['Melanoma','Nevi','Basal Cell Carcinoma',
                'Actinic Keratosis','Benign Keratosis',
                'Dermatofibroma','Vascular']

torch.manual_seed(42)
np.random.seed(42)
os.makedirs(OUTPUT,    exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"✓ Config ready")
print(f"  Device     : {DEVICE}")
print(f"  Model      : {TIMM_STR}")
print(f"  Batch      : {BATCH_SIZE} × {GRAD_ACCUM} accum = {BATCH_SIZE*GRAD_ACCUM} effective")
print(f"  LR         : {LR}  (lower than CNN — transformer needs it)")


## Cell 3 — Verify Paths (Diagnostic First)

In [ ]:
# Run this first to confirm all datasets are mounted
import os
print("=== ALL KAGGLE INPUT PATHS ===")
for root, dirs, files in os.walk('/kaggle/input'):
    depth = root.replace('/kaggle/input', '').count(os.sep)
    if depth > 3: continue
    indent = '  ' * depth
    print(f"{indent}{os.path.basename(root)}/  ({len(files)} files)")


## Cell 4 — Load ALL 6 Datasets & Remap Paths

In [ ]:
BASE   = '/kaggle/input/datasets/akshat23029'
HAM1   = f'{BASE}/dermograph-ham-images/HAM10000_images_part_1'
HAM2   = f'{BASE}/dermograph-ham-images/HAM10000_images_part_2'
PAD1   = f'{BASE}/dermograph-pad-images/images/imgs_part_1'
PAD2   = f'{BASE}/dermograph-pad-images/images/imgs_part_2'
PAD3   = f'{BASE}/dermograph-pad-images/images/imgs_part_3'
SPLITS = f'{BASE}/dermograph-splits'
ISIC   = f'{BASE}/dermograph-isic2020'
MELAN  = f'{BASE}/dermograph-melanoma-cancer'
MIDAS  = f'{BASE}/dermograph-midas/midasmultimodalimagedatasetforaibasedskincancer'

print("Checking dataset paths...")
all_ok = True
for name, path in [
    ('HAM part1',       HAM1),
    ('HAM part2',       HAM2),
    ('PAD part1',       PAD1),
    ('PAD part2',       PAD2),
    ('PAD part3',       PAD3),
    ('Splits CSV',      SPLITS),
    ('ISIC 2020',       ISIC),
    ('Melanoma Cancer', MELAN),
    ('MIDAS',           MIDAS),
]:
    exists = os.path.exists(path)
    n      = len([f for f in os.listdir(path) if not f.startswith('.')]) if exists else 0
    print(f"  {'✓' if exists else '✗'} {name:<20}: {n:,} files")
    if not exists: all_ok = False

assert all_ok, "❌ Missing datasets! Add via + Add Input on the right panel"

train_df = pd.read_csv(f'{SPLITS}/train.csv')
val_df   = pd.read_csv(f'{SPLITS}/val.csv')
test_df  = pd.read_csv(f'{SPLITS}/test.csv')
with open(f'{SPLITS}/class_weights.json') as f:
    cw_raw = json.load(f)

print(f"\nSplit sizes: Train={len(train_df):,}  Val={len(val_df):,}  Test={len(test_df):,}")

# Build lookups
pad_lookup = {}
for part in [PAD1, PAD2, PAD3]:
    for fname in os.listdir(part):
        if fname.lower().endswith(('.jpg','.jpeg','.png')):
            pad_lookup[fname] = f"{part}/{fname}"

isic_lookup = {}
for fname in os.listdir(ISIC):
    if fname.lower().endswith(('.jpg','.jpeg','.png')):
        isic_lookup[fname] = f"{ISIC}/{fname}"

midas_lookup = {}
for fname in os.listdir(MIDAS):
    if fname.lower().endswith(('.jpg','.jpeg','.png','.JPG','.JPEG')):
        midas_lookup[fname] = f"{MIDAS}/{fname}"

print(f"\nLookup tables: PAD={len(pad_lookup):,}  ISIC={len(isic_lookup):,}  MIDAS={len(midas_lookup):,}")

def remap(mac_path):
    p     = str(mac_path)
    fname = os.path.basename(p)
    if 'HAM10000_images_part_1' in p:
        fp = f"{HAM1}/{fname}"; return fp if os.path.exists(fp) else None
    if 'HAM10000_images_part_2' in p:
        fp = f"{HAM2}/{fname}"; return fp if os.path.exists(fp) else None
    if 'PAD-UFES' in p or 'pad' in p.lower() or 'imgs_part' in p:
        return pad_lookup.get(fname)
    if 'ISIC 2020' in p or 'isic' in p.lower():
        return isic_lookup.get(fname)
    if 'melanoma_cancer_dataset' in p:
        for split in ['train','test']:
            for cls in ['malignant','benign']:
                fp = f"{MELAN}/{split}/{cls}/{fname}"
                if os.path.exists(fp): return fp
        return None
    if 'MIDAS' in p or 'midas' in p.lower():
        return midas_lookup.get(fname)
    return None

print("\nRemapping paths...")
for name, df in [('train',train_df),('val',val_df),('test',test_df)]:
    df['kpath'] = df['image_path'].apply(remap)
    before = len(df)
    df.drop(df[df['kpath'].isna()].index, inplace=True)
    df.drop(df[df['label'] >= N_CLASSES].index, inplace=True)
    df.reset_index(drop=True, inplace=True)
    src = df['source'].value_counts().to_dict()
    print(f"  {name:<6}: {len(df):>6,} images (dropped {before-len(df)}) | {src}")

print("\n✓ All datasets loaded and paths remapped")


## Cell 5 — Cache Images (Resize Only)

In [ ]:
all_paths = list(set(
    train_df['kpath'].tolist() +
    val_df['kpath'].tolist()   +
    test_df['kpath'].tolist()
))

already = len([f for f in os.listdir(CACHE_DIR) if f.endswith('.jpg')])
print(f"Already cached : {already:,}")
print(f"Total needed   : {len(all_paths):,}")
print("Skips existing — fast if previous model cache exists\n")

done = skipped = errors = 0
for src_path in tqdm(all_paths, desc="Caching"):
    fname    = os.path.basename(src_path)
    uid      = str(abs(hash(src_path)))[:10]
    dst_path = f"{CACHE_DIR}{uid}_{fname}"
    if os.path.exists(dst_path):
        skipped += 1
        continue
    img = cv2.imread(src_path)
    if img is None:
        errors += 1
        continue
    img = cv2.resize(img, (224, 224), interpolation=cv2.INTER_LINEAR)
    cv2.imwrite(dst_path, img, [cv2.IMWRITE_JPEG_QUALITY, 95])
    done += 1

print(f"\n✓ Done: {done:,} new | {skipped:,} skipped | {errors} errors")

cache_lookup = {}
for fname in os.listdir(CACHE_DIR):
    uid = fname[:10]
    cache_lookup[uid] = f"{CACHE_DIR}{fname}"

def to_cache(p):
    uid = str(abs(hash(p)))[:10]
    return cache_lookup.get(uid, p)

for df in [train_df, val_df, test_df]:
    df['kpath'] = df['kpath'].apply(to_cache)

total, used, free = shutil.disk_usage(CACHE_DIR)
print(f"Cache: {used/1e9:.2f} GB | Free: {free/1e9:.1f} GB")
print("✓ Paths updated to cache")


## Cell 6 — Dataset & Augmentation

In [ ]:
def get_train_transforms():
    return A.Compose([
        A.RandomRotate90(p=0.5),
        A.Rotate(limit=180, p=0.7),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
        A.GaussianBlur(blur_limit=(3,7), p=0.3),
        A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.3),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

def get_val_transforms():
    return A.Compose([
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])

class DermDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(str(row['kpath']))
        if img is None:
            img = np.zeros((224,224,3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(image=img)['image']
        return img, torch.tensor(int(row['label']), dtype=torch.long)

img, lbl = DermDataset(train_df, get_train_transforms())[0]
print(f"✓ Dataset OK | img: {img.shape} | label: {lbl.item()} = {CLASS_NAMES[lbl.item()]}")


## Cell 7 — DataLoaders & Class Weights

In [ ]:
cw_list = [min(float(cw_raw.get(str(i), 1.0)), 10.0) for i in range(N_CLASSES)]
class_weights = torch.tensor(cw_list, dtype=torch.float32).to(DEVICE)

print("Class weights (capped at 10x):")
for i, (name, w) in enumerate(zip(CLASS_NAMES, cw_list)):
    print(f"  {i} {name:<25} {w:.2f}  {'█'*int(w*2)}")

train_labels = train_df['label'].values
sample_w     = np.array(cw_list)[train_labels]
sampler      = WeightedRandomSampler(
    weights     = torch.tensor(sample_w, dtype=torch.float32),
    num_samples = len(train_df),
    replacement = True,
)

train_loader = DataLoader(
    DermDataset(train_df, get_train_transforms()),
    batch_size  = BATCH_SIZE,
    sampler     = sampler,
    num_workers = 2,
    pin_memory  = True,
)
val_loader = DataLoader(
    DermDataset(val_df, get_val_transforms()),
    batch_size  = BATCH_SIZE * 2,
    shuffle     = False,
    num_workers = 2,
    pin_memory  = True,
)
test_loader = DataLoader(
    DermDataset(test_df, get_val_transforms()),
    batch_size  = BATCH_SIZE * 2,
    shuffle     = False,
    num_workers = 2,
    pin_memory  = True,
)

print(f"\n✓ DataLoaders ready")
print(f"  Train batches : {len(train_loader)}")
print(f"  Val   batches : {len(val_loader)}")
print(f"  Test  batches : {len(test_loader)}")


## Cell 8 — Build MaxViT-T Model

In [ ]:
model = timm.create_model(
    TIMM_STR,
    pretrained  = True,
    num_classes = N_CLASSES,
)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"✓ MaxViT-T loaded")
print(f"  timm string  : {TIMM_STR}")
print(f"  Total params : {total_params/1e6:.1f}M")
print(f"  Classes      : {N_CLASSES}")

# Quick forward pass test
with torch.no_grad():
    dummy = torch.randn(2, 3, 224, 224).to(DEVICE)
    out   = model(dummy)
print(f"  Output shape : {out.shape}  ✓")


## Cell 9 — Loss, Optimizer, Scheduler

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = AdamW(
    model.parameters(),
    lr           = LR,
    weight_decay = WEIGHT_DECAY,
)
scheduler = CosineAnnealingLR(
    optimizer,
    T_max   = N_EPOCHS,
    eta_min = 1e-7,
)
scaler = GradScaler()

print("✓ Loss / Optimizer / Scheduler ready")
print(f"  Loss      : CrossEntropyLoss + class weights")
print(f"  Optimizer : AdamW (lr={LR}, wd={WEIGHT_DECAY})")
print(f"  Scheduler : CosineAnnealingLR (T_max={N_EPOCHS}, eta_min=1e-7)")


## Cell 10 — Evaluate Function (FP32 + row-normalized AUC)

In [ ]:
def evaluate(model, loader, desc='Evaluating'):
    model.eval()
    va_loss_sum = 0.0
    all_preds   = []
    all_labels  = []
    all_probs   = []

    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc=desc, leave=False, ncols=90):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out  = model(imgs).float()          # force FP32 — fixes AUC=0 bug
            loss = criterion(out, labels)
            va_loss_sum += loss.item()
            probs = torch.softmax(out, dim=1).cpu().numpy().astype(np.float64)
            preds = out.argmax(dim=1).cpu().numpy()
            all_probs.append(probs)
            all_preds.append(preds)
            all_labels.append(labels.cpu().numpy())

    all_probs  = np.vstack(all_probs).astype(np.float64)
    all_preds  = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    # Row-normalize to guarantee sum=1.0 (required by roc_auc_score)
    all_probs  = all_probs / all_probs.sum(axis=1, keepdims=True)

    va_loss = va_loss_sum / len(loader)
    va_acc  = float((all_preds == all_labels).mean())
    va_f1   = float(f1_score(all_labels, all_preds, average='macro', zero_division=0))
    try:
        va_auc = float(roc_auc_score(
            all_labels, all_probs,
            multi_class='ovr', average='macro'
        ))
    except Exception as e:
        print(f"  [AUC err]: {e}")
        va_auc = 0.0

    return va_loss, va_acc, va_f1, va_auc, all_preds, all_labels, all_probs

print("✓ evaluate() ready — FP32 forced, rows normalized")


## Cell 11 — TRAIN MaxViT-T (50 Epochs)
> Expected: ~3-4 min/epoch on T4 → ~150-200 min total

In [ ]:
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"Disk before training: {used/1e9:.1f}GB used / {free/1e9:.1f}GB free\n")

CKPT_PATH    = f"{OUTPUT}{MODEL_NAME}_best.pth"
best_val_acc = 0.0
best_val_auc = 0.0
no_improve   = 0
history      = []
start_total  = time.time()

print(f"Training MAXVIT-T — {N_EPOCHS} epochs max, patience={PATIENCE}")
print(f"Saving: {MODEL_NAME}_best.pth only (no per-epoch saves)")
print("=" * 85)
print(" Ep | TrainLoss |  ValLoss |  ValAcc |      F1 |     AUC |   Min | Status")
print("=" * 85)
sys.stdout.flush()

for epoch in range(N_EPOCHS):

    # ── Train ──────────────────────────────────────────────
    model.train()
    tr_loss_sum = 0.0
    optimizer.zero_grad()

    train_bar = tqdm(
        enumerate(train_loader),
        total  = len(train_loader),
        desc   = f"Ep {epoch+1:>2}/{N_EPOCHS} [Train]",
        leave  = False,
        ncols  = 90,
    )
    for step, (imgs, labels) in train_bar:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast():
            loss = criterion(model(imgs), labels) / GRAD_ACCUM
        scaler.scale(loss).backward()
        if (step+1) % GRAD_ACCUM == 0 or (step+1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        tr_loss_sum += loss.item() * GRAD_ACCUM
        train_bar.set_postfix(loss=f"{loss.item()*GRAD_ACCUM:.4f}")

    tr_loss = tr_loss_sum / len(train_loader)

    # ── Validate ───────────────────────────────────────────
    va_loss, va_acc, va_f1, va_auc, _, _, _ = evaluate(
        model, val_loader,
        desc=f"Ep {epoch+1:>2}/{N_EPOCHS} [Val]  "
    )

    scheduler.step()
    elapsed_min = (time.time() - start_total) / 60

    # ── Checkpoint ─────────────────────────────────────────
    if va_acc > best_val_acc:
        best_val_acc = va_acc
        best_val_auc = va_auc
        torch.save(model.state_dict(), CKPT_PATH)
        no_improve = 0
        status = "⭐ BEST"
    else:
        no_improve += 1
        status = f"pat {no_improve}/{PATIENCE}"

    print(f"{epoch+1:>3} | {tr_loss:>9.4f} | {va_loss:>8.4f} | "
          f"{va_acc:>7.4f} | {va_f1:>7.4f} | {va_auc:>7.4f} | "
          f"{elapsed_min:>5.1f} | {status}")
    sys.stdout.flush()

    history.append(dict(
        epoch=epoch+1, tr_loss=tr_loss,
        va_loss=va_loss, va_acc=va_acc,
        va_f1=va_f1, va_auc=va_auc,
    ))

    if no_improve >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

total_min = (time.time() - start_total) / 60
print("=" * 85)
print(f"Training done in {total_min:.1f} min")
print(f"  Best Val Acc : {best_val_acc*100:.2f}%")
print(f"  Best Val AUC : {best_val_auc:.4f}")
print(f"  Epochs run   : {len(history)}")
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"  Disk used    : {used/1e9:.1f}GB / {free/1e9:.1f}GB free")

with open(f"{OUTPUT}{MODEL_NAME}_history.json", 'w') as f:
    json.dump(history, f, indent=2)
print("✓ History saved")


## Cell 12 — Training Curves

In [ ]:
hist_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('MaxViT-T — Training Curves', fontsize=14, fontweight='bold')

axes[0].plot(hist_df['epoch'], hist_df['tr_loss'], label='Train Loss', color='#e74c3c')
axes[0].plot(hist_df['epoch'], hist_df['va_loss'], label='Val Loss',   color='#2ecc71')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(hist_df['epoch'], hist_df['va_acc'], label='Val Acc', color='#2ecc71')
axes[1].axhline(y=best_val_acc, color='gold', linestyle='--', label=f'Best {best_val_acc:.4f}')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(hist_df['epoch'], hist_df['va_auc'], label='Val AUC', color='#3498db')
axes[2].axhline(y=best_val_auc, color='gold', linestyle='--', label=f'Best {best_val_auc:.4f}')
axes[2].set_title('AUC-ROC'); axes[2].set_xlabel('Epoch')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT}{MODEL_NAME}_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {MODEL_NAME}_curves.png")


## Cell 13 — Final Test Evaluation

In [ ]:
print("Loading best checkpoint...")
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))

ts_loss, ts_acc, ts_f1, ts_auc, ts_preds, ts_labels, ts_probs = evaluate(
    model, test_loader, desc="Testing"
)

print("\n" + "=" * 58)
print(f"  MAXVIT-T — FINAL TEST RESULTS")
print("=" * 58)
print(f"  Test Accuracy : {ts_acc*100:.2f}%")
print(f"  Test F1 Macro : {ts_f1:.4f}")
print(f"  Test AUC-ROC  : {ts_auc:.4f}")
print(f"  Best Val Acc  : {best_val_acc*100:.2f}%")
print(f"  Best Val AUC  : {best_val_auc:.4f}")
print(f"  Epochs run    : {len(history)}")
print("=" * 58)

print("\nPer-class Report:")
print(classification_report(ts_labels, ts_preds, target_names=CLASS_NAMES, digits=2))

print("\n" + "=" * 58)
print("  BENCHMARK COMPARISON")
print("=" * 58)
print(f"  {'Model':<20} {'Acc':>8} {'F1':>8} {'AUC':>8}")
print(f"  {'-'*20} {'-'*8} {'-'*8} {'-'*8}")
print(f"  {'VGG16':<20} {'80.48%':>8} {'0.7102':>8} {'0.9601':>8}")
print(f"  {'ResNet50':<20} {'87.40%':>8} {'0.7261':>8} {'0.9823':>8}")
print(f"  {'DenseNet121':<20} {'87.69%':>8} {'0.7663':>8} {'0.9866':>8}")
print(f"  {'MaxViT-T ★':<20} {ts_acc*100:>7.2f}% {ts_f1:>8.4f} {ts_auc:>8.4f}")
print("=" * 58)


## Cell 14 — Confusion Matrix (Viridis Style)

In [ ]:
cm    = confusion_matrix(ts_labels, ts_preds)
SHORT = ['MEL','NV','BCC','AK','BKL','DF','VASC']

# Raw counts
fig, ax = plt.subplots(figsize=(9,7))
im   = ax.imshow(cm, cmap='viridis', aspect='auto')
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.ax.tick_params(labelsize=11)
for i in range(len(SHORT)):
    for j in range(len(SHORT)):
        color = 'white' if cm[i,j] < cm.max()*0.6 else 'black'
        ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                fontsize=14, fontweight='bold', color=color)
ax.set_xticks(np.arange(len(SHORT))); ax.set_yticks(np.arange(len(SHORT)))
ax.set_xticklabels(SHORT, fontsize=12, fontweight='bold')
ax.set_yticklabels(SHORT, fontsize=12, fontweight='bold')
ax.set_xlabel('Predicted label', fontsize=13, fontweight='bold', labelpad=10)
ax.set_ylabel('True label',      fontsize=13, fontweight='bold', labelpad=10)
ax.set_title('MaxViT-T (DL Only)', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(np.arange(len(SHORT))-0.5, minor=True)
ax.set_yticks(np.arange(len(SHORT))-0.5, minor=True)
ax.grid(which='minor', color='white', linewidth=2)
ax.tick_params(which='minor', length=0)
plt.tight_layout()
plt.savefig(f"{OUTPUT}{MODEL_NAME}_cm_viridis.png", dpi=150, bbox_inches='tight')
plt.show()

# Normalized
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(9,7))
im   = ax.imshow(cm_norm, cmap='viridis', aspect='auto', vmin=0, vmax=1)
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.ax.tick_params(labelsize=11)
for i in range(len(SHORT)):
    for j in range(len(SHORT)):
        color = 'white' if cm_norm[i,j] < 0.6 else 'black'
        ax.text(j, i, f'{cm_norm[i,j]:.2f}', ha='center', va='center',
                fontsize=13, fontweight='bold', color=color)
ax.set_xticks(np.arange(len(SHORT))); ax.set_yticks(np.arange(len(SHORT)))
ax.set_xticklabels(SHORT, fontsize=12, fontweight='bold')
ax.set_yticklabels(SHORT, fontsize=12, fontweight='bold')
ax.set_xlabel('Predicted label', fontsize=13, fontweight='bold', labelpad=10)
ax.set_ylabel('True label',      fontsize=13, fontweight='bold', labelpad=10)
ax.set_title('MaxViT-T (DL Only) — Normalized', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(np.arange(len(SHORT))-0.5, minor=True)
ax.set_yticks(np.arange(len(SHORT))-0.5, minor=True)
ax.grid(which='minor', color='white', linewidth=2)
ax.tick_params(which='minor', length=0)
plt.tight_layout()
plt.savefig(f"{OUTPUT}{MODEL_NAME}_cm_normalized.png", dpi=150, bbox_inches='tight')
plt.show()
print("✓ Confusion matrices saved")


## Cell 15 — TP/FP/FN/TN per Class (Dark Theme)

In [ ]:
COL_TP = '#2ecc71'
COL_FN = '#e74c3c'
COL_FP = '#e6a817'
COL_TN = '#1abc9c'
BG     = '#1a1a2e'

total_n = len(ts_labels)
per = []
for i in range(N_CLASSES):
    TP = int(cm[i,i])
    FN = int(cm[i,:].sum() - TP)
    FP = int(cm[:,i].sum() - TP)
    TN = int(total_n - TP - FN - FP)
    per.append((TP, FN, FP, TN, int(cm[i,:].sum())))

rows_idx = [0,0,0,0,1,1,1]
cols_idx = [0,1,2,3,0,1,2]
n_cols   = 4

fig = plt.figure(figsize=(22,14), facecolor=BG)
fig.patch.set_facecolor(BG)
fig.text(0.5, 0.97, 'MAXVIT-T — TP/FP/FN/TN per Class',
         ha='center', va='top', fontsize=18, fontweight='bold', color='white')
fig.text(0.5, 0.93, f'Acc={ts_acc*100:.2f}%   F1={ts_f1:.4f}   AUC={ts_auc:.4f}',
         ha='center', va='top', fontsize=14, fontweight='bold', color='white')

left_margin = 0.04
right_margin= 0.78
top_margin  = 0.88
row_height  = 0.38
col_width   = (right_margin - left_margin) / n_cols
cell_w      = col_width  * 0.42
cell_h      = row_height * 0.42

def draw_block(fig, cx, cy, tp, fn, fp, tn, cname, support):
    fig.text(cx+col_width/2, cy+row_height*0.96, cname,
             ha='center', va='top', fontsize=11, fontweight='bold', color='white')
    fig.text(cx+col_width/2, cy+row_height*0.89, f'(n={support})',
             ha='center', va='top', fontsize=9, color='#aaaaaa')
    pad_x = col_width  * 0.04
    pad_y = row_height * 0.06
    gap   = col_width  * 0.02
    for qc, qr, val, color, label in [
        (0, 1, tp, COL_TP, 'TP'),
        (1, 1, fn, COL_FN, 'FN'),
        (0, 0, fp, COL_FP, 'FP'),
        (1, 0, tn, COL_TN, 'TN'),
    ]:
        x  = cx + pad_x + qc*(cell_w+gap)
        y  = cy + pad_y + qr*(cell_h+gap)
        ax = fig.add_axes([x, y, cell_w, cell_h])
        ax.set_facecolor(color)
        ax.set_xlim(0,1); ax.set_ylim(0,1)
        for sp in ax.spines.values(): sp.set_visible(False)
        ax.set_xticks([]); ax.set_yticks([])
        ax.text(0.5,0.58,str(val), ha='center', va='center',
                fontsize=20, fontweight='bold', color='white', transform=ax.transAxes)
        ax.text(0.5,0.18,label, ha='center', va='center',
                fontsize=11, fontweight='bold', color='white', transform=ax.transAxes)
    fig.text(cx+pad_x-0.005, cy+pad_y+1.5*cell_h+gap,
             'Act\nNeg', ha='right', va='center', fontsize=7.5, color='#cccccc')
    fig.text(cx+pad_x-0.005, cy+pad_y+0.5*cell_h,
             'Act\nPos', ha='right', va='center', fontsize=7.5, color='#cccccc')
    fig.text(cx+pad_x+0.5*cell_w, cy+pad_y-0.025,
             'Pred\nPos', ha='center', va='top', fontsize=7.5, color='#cccccc')
    fig.text(cx+pad_x+1.5*cell_w+gap, cy+pad_y-0.025,
             'Pred\nNeg', ha='center', va='top', fontsize=7.5, color='#cccccc')

for i, (cname, (tp,fn,fp,tn,sup)) in enumerate(zip(CLASS_NAMES, per)):
    cx = left_margin + cols_idx[i]*col_width
    cy = top_margin  - (rows_idx[i]+1)*row_height + 0.02
    draw_block(fig, cx, cy, tp, fn, fp, tn, cname, sup)

# Legend
leg_x = 0.80; leg_y = 0.55
fig.text(leg_x+0.04, leg_y+0.06, 'Legend',
         ha='center', va='bottom', fontsize=13, fontweight='bold', color='white')
for k, (color, label, desc) in enumerate([
    (COL_TP,'TP','Correctly detected'),
    (COL_FN,'FN','Missed / not detected'),
    (COL_FP,'FP','Wrongly detected'),
    (COL_TN,'TN','Correctly rejected'),
]):
    y = leg_y - k*0.07
    ax_leg = fig.add_axes([leg_x, y-0.025, 0.035, 0.045])
    ax_leg.set_facecolor(color)
    ax_leg.set_xticks([]); ax_leg.set_yticks([])
    for sp in ax_leg.spines.values(): sp.set_visible(False)
    ax_leg.text(0.5,0.5,label, ha='center', va='center',
                fontsize=10, fontweight='bold', color='white', transform=ax_leg.transAxes)
    fig.text(leg_x+0.045, y, desc, ha='left', va='center', fontsize=10, color='#cccccc')

plt.savefig(f"{OUTPUT}{MODEL_NAME}_tp_fp_fn_tn.png",
            dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print(f"✓ Saved: {MODEL_NAME}_tp_fp_fn_tn.png")


## Cell 16 — Save Results JSON & Final Summary

In [ ]:
per_precision = precision_score(ts_labels, ts_preds, average=None, zero_division=0)
per_recall    = recall_score(ts_labels, ts_preds, average=None, zero_division=0)
per_f1_vals   = f1_score(ts_labels, ts_preds, average=None, zero_division=0)

results = {
    'model'         : MODEL_NAME,
    'timm_str'      : TIMM_STR,
    'params_M'      : round(sum(p.numel() for p in model.parameters())/1e6, 1),
    'test_accuracy' : float(ts_acc),
    'test_f1_macro' : float(ts_f1),
    'test_auc_roc'  : float(ts_auc),
    'best_val_acc'  : float(best_val_acc),
    'best_val_auc'  : float(best_val_auc),
    'epochs_run'    : len(history),
    'batch_size'    : BATCH_SIZE,
    'lr'            : LR,
    'per_class'     : {
        name: {
            'precision': float(per_precision[i]),
            'recall'   : float(per_recall[i]),
            'f1'       : float(per_f1_vals[i]),
        } for i, name in enumerate(CLASS_NAMES)
    },
    'history': history,
}

with open(f"{OUTPUT}{MODEL_NAME}_results.json", 'w') as f:
    json.dump(results, f, indent=2)

torch.cuda.empty_cache()
total, used, free = shutil.disk_usage("/kaggle/working")

print("=" * 58)
print(f"  ✓ MAXVIT-T COMPLETE")
print("=" * 58)
print(f"  Test Acc  : {ts_acc*100:.2f}%")
print(f"  Test F1   : {ts_f1:.4f}")
print(f"  Test AUC  : {ts_auc:.4f}")
print(f"  Disk used : {used/1e9:.1f}GB / {free/1e9:.1f}GB free")
print("=" * 58)
print("\n  ⚠️  SAVE VERSION NOW before closing!")
print("  Click 'Save Version' → 'Save and Run All'")
print("\n  Files to download from Output tab:")
for fname in sorted(os.listdir(OUTPUT)):
    if MODEL_NAME in fname and not fname.endswith('.ipynb'):
        sz = os.path.getsize(f'{OUTPUT}/{fname}')/1024
        tag = f"{sz:.0f} KB" if sz < 1024 else f"{sz/1024:.1f} MB"
        print(f"    {fname:<45} {tag}")
print("\n  FINAL BENCHMARK:")
print(f"  VGG16       80.48%  F1=0.7102  AUC=0.9601")
print(f"  ResNet50    87.40%  F1=0.7261  AUC=0.9823")
print(f"  DenseNet121 87.69%  F1=0.7663  AUC=0.9866")
print(f"  MaxViT-T    {ts_acc*100:.2f}%  F1={ts_f1:.4f}  AUC={ts_auc:.4f}")
